# Instructions
Before you start this lesson you need to set up Google Colab a bit more:

1. Click the Runtime > Change runtime type menu option at the top of the page:
   ![Screenshot of change runtime type menu](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/colab-change-runtime-type-menu.png?raw=true)
2. Then select "T4 GPU" under the hardware accelerator section and click "Save"
   ![Change runtime type popup screenshot](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/colab-change-runtime-type-popup.png?raw=true)
3. Finally Click the "Run all" button at the top of the page:
   ![Screenshot showing run all button](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/run-all-button.png?raw=true)
4. If you completed these instructions correctly you should see this message a few inches below on this page:
   ![Screenshot of image showing pytorch setup was successful](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/colab-pytorch-accel-found.png?raw=true)
   If you don't see this message please verify you completed steps 1 and 2 correctly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, interactive_output
import ipywidgets as widgets
import IPython.display as display
import torch
from torch import nn
from torch.utils.data import Dataset, random_split
import graphviz
import plotly.graph_objects as go
from pprint import pprint

In [ ]:
# @title
if torch.accelerator.is_available():
    print("Excellent, you followed the setup steps correctly")
else:
    print("Whoops! It looks like Google Colab isn't setup correctly, please review the setup instructions above")

# Neural Networks
So far we've just been learning about mathematical modeling. This is the basis neural networks are built on.

So what are neural networks?

- They are modeled on the way brains work
- Made up of a bunch of **neurons**
- Neurons are highly connected to other neurons

<img alt="Image of neurons in the brain" src="https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/neurons.webp?raw=true" width="800px" />

Let's start at the smallest building block, a neuron, and then work our way up to a full network.

## Neurons
### A Familiar Equation
Remember the equation we were using earlier?  

$y = m \cdot x + b$

Neurons are actually shockingly similar to this equation. The most simple neuron with one input has the equation:

$y = tanh(w \cdot x + b)$

A neuron with 2 inputs would have the equation:

$y = tanh((w_1 \cdot x_1) + (w_2 \cdot x_2) + b))$

The whole equation is very similar:

- We still have slopes (except instead of calling them $m$ we call them $w$)
- We still have input $x$ values (except we can have more than 1)
- We still have a $b$ value which we add at the end

The biggest difference is that we wrap the entire output in $tanh$, we will talk about that more later.

Since we can have multiple inputs (multiple $x$'s) and multiple slopes (multiple $w$'s) we need a way of telling the difference between them:

- To do this we put a number after the variable name like so: $x_1$ or $w_1$
- $x_1$ means this is the first $x$ value
- $w_1$ means this is the first $w$ value
- $x_{10}$ means this is the tenth $x$ value
- Sometimes we just write this as $x1$ or $w1$, it means the same thing

### Neuron In Depth
Let's dig into the detail of a neuron. Here is a diagram of a neuron with 4 inputs:

![Image of a single neuron](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/single-neuron.png?raw=true)

A neuron is composed of:

- Inputs ($x_i$): Values fed in to the neuron, these are the outputs from other neurons
- Weights ($w_i$): Input values are multiplied each by their own weight ($x_i \cdot w_i$)
  - These are the values we will change to train our neuron

Say a neuron has 4 inputs ($x_1$, $x_2$, $x_3$, and $x_4$). It will also have the same number of weights ($w_1$, $w_2$, $w_3$, and $w_4$).

These inputs and weights are multiplied and added up like so:  
$(w_1 \cdot x_1) + (w_2 \cdot x_2) + (w_3 \cdot x_3) + (w_4 \cdot x_4)$

Neurons also have:

- Bias ($b$): Added to the sum of all the inputs and their weights
  - We also change this value to train our neuron
- Activation function ($a$): The sum of all the inputs plus the bias are run through this function
  - This result is used as the output of the neuron
  - This function is how the neuron "decides" if all the inputs big enough to output
  - These are just normal math functions which take a single input and output a number

#### Activation Functions
The activation function used in neurons are pretty important. They are the math which takes in all the inputs and decides what the output of the neuron should be.

A pretty common activation function is $tanh$. To get familiar with $tanh$ let's see what it looks like on a graph:

In [ ]:
# @title
x = np.arange(-5, 5, 0.1)
y = np.tanh(x)

plt.plot(x, y)
plt.grid()
plt.title("tanh")
plt.gca().set_xlabel("inputs + bias")
plt.gca().set_ylabel("output")
plt.gca().set_xticks(np.arange(-5, 6, 1))
plt.axvline(0, color='black')
plt.axhline(0, color='black')
plt.show()

$tanh$ has some useful properties:

- The value will always be between $-1$ and $1$ (inclusive)
- It doesn't get overwhelmed by extremely large or extremely small inputs
  - Instead it just outputs $1$ or $-1$ respectively in these cases
- Preserves middle value inputs
  | Input | $tanh$ |
  | - | - |
  | -1.5 | -0.91 |
  | -1 | -0.76 |
  | -0.75 | -0.64 |
  | -0.5 | -0.42 |
  | 0.5 | 0.42 |
  | 0.75 | 0.64 |
  | 1 | 0.76 |
  | 1.5 | 0.91 |

Two other common activation functions are "ReLU" and "Sigmoid". They are similar to $tanh$, just with more scary sounding names.

First let's look at "Sigmoid" on a graph:

In [ ]:
# @title
x = torch.arange(-5, 5, 0.1)
y_sigmoid = torch.sigmoid(x)

plt.plot(x, y_sigmoid)
plt.grid()
plt.gca().set_xlim(-5, 6)
plt.gca().set_ylim(0, 1)
plt.axvline(0, color='black')
plt.gca().set_xlabel("inputs + bias")
plt.gca().set_ylabel("output")
plt.title("Sigmoid")

plt.show()


You can see Sigmoid is similar to $tanh$:

- If a value is to big or small it just outputs $0$ or $1$
- Preserves values in the middle

The only difference is that $tanh$ deals with values between $-1$ and $1$. But Sigmoid deals with values between $0$ and $1$.

Now let's look at ReLU on a graph:

In [ ]:
# @title
x = torch.arange(-5, 5, 0.1)
y_relu = torch.relu(x)

plt.plot(x, y_relu)
plt.grid()
plt.gca().set_xlim(-5, 5)
plt.gca().set_ylim(-5, 5)
plt.gca().set_xticks(np.arange(-5, 6, 1))
plt.gca().set_yticks(np.arange(-5, 6, 1))
plt.axvline(0, color='black')
plt.gca().set_xlabel("inputs + bias")
plt.gca().set_ylabel("output")
plt.title("ReLu")

plt.show()

ReLU is totally different:

- If the input value is negative ReLU returns $0$
- If the input value is positive ReLU returns exactly the input number

### Comparing Activation Functions
There is a lot of thought which goes into choosing the activation function a neural network uses. But as a basic rule:

- If you want your output values to be between $-1$ and $1$ use $tanh$
- If you want your output values to be between $0$ and $1$ use Sigmoid
- If you don't want negative numbers use ReLU

We'll stick with $tanh$ for the remainder of this section. But we could just as well used Sigmoid or ReLU.

# Check In - Neurons
We just learned about neurons. Here are the key points you should understand:

- Neurons are very similar to our $y = m \cdot x + b$ function
  - Except they can have multiple $m$ and $x$ values
- Neurons have inputs which are numbers
- Neurons have weights which are multiplied times inputs
  - We tune these weights to get the neuron to do what we want
- Neurons have a bias which is added to the weights and inputs
  - We tune this bias to get the neuron to do what we want
- Neurons have an activation function
  - We take the result of the inputs, weights, and bias and run it through this activation function

## Interactive Neuron Introduction
In the following sections you will be able to tweak the inputs, weights, and bias of a single neuron to see how the output changes.

The interactive neuron is shown in a bit more detail, and it looks more confusing than the picture above:

In [ ]:
# @title
def visualize_neuron(x1=0.5, x2=0.3, w1=0.8, w2=-0.4, b=0.2):
    """
    Visualize a single neuron with interactive sliders.

    Args:
        x1, x2: Input values
        w1, w2: Weight values
        b: Bias value
    """
    # Calculate neuron values
    input1_weighted = x1 * w1
    input2_weighted = x2 * w2
    weighted_sum = input1_weighted + input2_weighted
    total_with_bias = weighted_sum + b

    # Simple tanh activation (approximate calculation)
    activation_output = np.tanh(total_with_bias)

    # Create the visualization
    dot = graphviz.Digraph("Neuron", format="png")
    dot.attr(rankdir="LR", size="8,5")
    dot.attr("node", shape="circle")

    # Input nodes
    dot.node("x1", f"{{ x1 | {x1:.2f} }}", shape="record")
    dot.node("x2", f"{{ x2 | {x2:.2f} }}", shape="record")

    # Weight edges from inputs to multiplication nodes
    dot.edge("x1", "mult1", label=f"w1={w1:.2f}")
    dot.edge("x2", "mult2", label=f"w2={w2:.2f}")

    with dot.subgraph(name="cluster_neuron") as sub_dot:
        sub_dot.attr(style='rounded', color='black', label='', shape='circle')

        # Multiplication nodes
        sub_dot.node("mult1", f"{{ x1*w1 | {input1_weighted:.2f} }}", shape="record")
        sub_dot.node("mult2", f"{{ x2*w2 | {input2_weighted:.2f} }}", shape="record")

        # Sum
        sub_dot.node("sum", f"{{ inputs sum | {weighted_sum:.2f} }}", shape="record")

        # Bias node
        sub_dot.node("bias", f"{{ b | {b:.2f} }}", shape="record")

        # Total sum node (with bias)
        sub_dot.node("total", f"{{ total sum | {total_with_bias:.2f} }}", shape="record")

    # Activation node
    dot.node("activation", f"tanh", shape="diamond")
    dot.node("output", f"{{ output | {activation_output:.2f} }}", shape="record")

    # Edges for multiplication results to sum
    dot.edge("mult1", "sum")
    dot.edge("mult2", "sum")

    # Edge from sum to total with bias
    dot.edge("sum", "total", label="sum")
    dot.edge("bias", "total", label="bias")

    # Edge from total to activation
    dot.edge("total", "activation")

    # Output label
    dot.edge("activation", "output")

    display.display(dot)

visualize_neuron()


![Image of a single neuron](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/single-neuron.png?raw=true)

In our new diagram:

- Inputs:  
  ![Image of inputs in new diagram](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-inputs.png?raw=true)
  - The inputs are on the left in boxes labeled $x1$ and $x2$
  - The values of $x1$ and $x2$ are on the right sides of the boxes
- Weights:  
  ![Image of weights in new diagram](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-weights.png?raw=true)
    - The weights are on arrows going out to the right of the input boxes
    - Labeled $w1$ and $w2$
- Result of $\text{inputs} \times \text{weights}$:  
  ![Image of inputs times weights](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-inputs-mult-weights.png?raw=true)
  - The inputs and weights multiplied together are shown in boxes at the end of the weights arrow
  - Labeled as $x1*w1$ and $x2*w2$
- Sum of both inputs and weights:  
  ![Inputs and weights summed](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-inputs-sum.png?raw=true)
  - The sum of both inputs and weights multiplied are in a box to the right
  - Labeled "inputs sum"
- Bias:  
  ![Bias box](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-bias.png?raw=true)
  - The bias is shown inside a box labeled "b"
- Input to activation function:  
  ![Input to activation function](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-bias-inputs.png?raw=true)
  - The total sum of the inputs multiplied weights plus the bias are shown in a box to the right
  - Labeled "total sum"
  - This is the input to the activation function
- Output of the neuron:  
  ![Output of the neuron](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/new-diagram-output.png?raw=true)
  - The output of the activation function, which is the output of the neuron, is shown farthest on the right
  - In a box labeled "output"

## Neuron - Intuition
Now that we know the definitions of each part of a neuron, let's start to think about what their purposes are.

### Inputs and Weights
Each input is the output of another neuron.

The weights tell the neuron how much to pay attention to each input.

Imagine we are measuring how much blue, red, and green is in an image. And those are the inputs to a neuron.

- If we are building a neural network that cares a lot about red the weight for red will probably be high
- While the weights for blue and green will probably be low

During the training of a neural network we try to find weight values which result in the lowest (best) error values (just like $m$ and $b$ earlier).

#### You Try - Neuron Weights
Play with this interactive neuron below. See how the output of the neuron changes just by changing the weights.

Find two different weight values which make the output of the neuron equal $0.3$.

In [ ]:
# @title
def visualize_neuron(x1=0.5, x2=0.3, w1=0.8, w2=-0.4, b=0.2):
    """
    Visualize a single neuron with interactive sliders.

    Args:
        x1, x2: Input values
        w1, w2: Weight values
        b: Bias value
    """
    # Calculate neuron values
    input1_weighted = x1 * w1
    input2_weighted = x2 * w2
    weighted_sum = input1_weighted + input2_weighted
    total_with_bias = weighted_sum + b

    # Simple tanh activation (approximate calculation)
    activation_output = np.tanh(total_with_bias)

    # Create the visualization
    dot = graphviz.Digraph("Neuron", format="png")
    dot.attr(rankdir="LR", size="8,5")
    dot.attr("node", shape="circle")

    # Input nodes
    dot.node("x1", f"{{ x1 | {x1:.2f} }}", shape="record")
    dot.node("x2", f"{{ x2 | {x2:.2f} }}", shape="record")

    # Weight edges from inputs to multiplication nodes
    dot.edge("x1", "mult1", label=f"w1={w1:.2f}")
    dot.edge("x2", "mult2", label=f"w2={w2:.2f}")

    with dot.subgraph(name="cluster_neuron") as sub_dot:
        sub_dot.attr(style='rounded', color='black', label='', shape='circle')

        # Multiplication nodes
        sub_dot.node("mult1", f"{{ x1*w1 | {input1_weighted:.2f} }}", shape="record")
        sub_dot.node("mult2", f"{{ x2*w2 | {input2_weighted:.2f} }}", shape="record")

        # Sum
        sub_dot.node("sum", f"{{ inputs sum | {weighted_sum:.2f} }}", shape="record")

        # Bias node
        sub_dot.node("bias", f"{{ b | {b:.2f} }}", shape="record")

        # Total sum node (with bias)
        sub_dot.node("total", f"{{ total sum | {total_with_bias:.2f} }}", shape="record")

    # Activation node
    dot.node("activation", f"tanh", shape="diamond")
    dot.node("output", f"{{ output | {activation_output:.2f} }}", shape="record")

    # Edges for multiplication results to sum
    dot.edge("mult1", "sum")
    dot.edge("mult2", "sum")

    # Edge from sum to total with bias
    dot.edge("sum", "total", label="sum")
    dot.edge("bias", "total", label="bias")

    # Edge from total to activation
    dot.edge("total", "activation")

    # Output label
    dot.edge("activation", "output")

    display.display(dot)

w1_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.5, description='w1')
w2_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.8, description='w2')
input1_box = widgets.HBox([w1_slider, w2_slider], layout={'border': '1px solid gray'})

# Organize into input group and bias group
inputs_group = widgets.VBox(
    [
        input1_box,
    ],
    layout={
        'margin-left': '200px',
    },
)

# Final layout
display.display(widgets.HBox([
    inputs_group,
]))

display.display(interactive_output(
    visualize_neuron,
    {
        "w1": w1_slider,
        "w2": w2_slider,
    }
))



### Bias
The bias is added to the sum of the inputs times their weights.

The bias shifts the output of the neuron to be more positive or negative generally.

The bias is important when the inputs and outputs aren't centered around $0$.

For example imagine our neuron is responsible for grading tests with scores from $0$ to $1$ (inclusive). To pass a test you need to get at least a $0.7$.

- The output of the neuron indicates if a student passed or failed
  - Passed: Output is above or equal to $0.5$
  - Failed: Output is below $0.5$
- The neuron has one input which is the test score

Just using weights (which are multiplied by the input) we cannot get the neuron to make the right decision. Try it with the neuron below:

In [ ]:
# @title
def visualize_neuron(x1=0.5, w1=0.8, b=0.0):
    """
    Visualize a single neuron with interactive sliders.

    Args:
        x1: Input values
        w1: Weight values
        b: Bias value
    """
    # Calculate neuron values
    input1_weighted = x1 * w1
    weighted_sum = input1_weighted
    total_with_bias = weighted_sum + b

    # Simple tanh activation (approximate calculation)
    activation_output = np.tanh(total_with_bias)

    # Create the visualization
    dot = graphviz.Digraph("Neuron", format="png")
    dot.attr(rankdir="LR", size="8,5")
    dot.attr("node", shape="circle")

    # Input nodes
    dot.node("x1", f"{{ x1 | {x1:.2f} }}", shape="record")

    # Weight edges from inputs to multiplication nodes
    dot.edge("x1", "mult1", label=f"w1={w1:.2f}")

    with dot.subgraph(name="cluster_neuron") as sub_dot:
        sub_dot.attr(style='rounded', color='black', label='', shape='circle')

        # Multiplication nodes
        sub_dot.node("mult1", f"{{ x1*w1 | {input1_weighted:.2f} }}", shape="record")

        # Sum
        sub_dot.node("sum", f"{{ inputs sum | {weighted_sum:.2f} }}", shape="record")

        # Bias node
        sub_dot.node("bias", f"{{ b | {b:.2f} }}", shape="record")

        # Total sum node (with bias)
        sub_dot.node("total", f"{{ total sum | {total_with_bias:.2f} }}", shape="record")

    # Activation node
    dot.node("activation", f"tanh", shape="diamond")
    dot.node("output", f"{{ output | {activation_output:.2f} }}", shape="record")

    # Edges for multiplication results to sum
    dot.edge("mult1", "sum")

    # Edge from sum to total with bias
    dot.edge("sum", "total", label="sum")
    dot.edge("bias", "total", label="bias")

    # Edge from total to activation
    dot.edge("total", "activation")

    # Output label
    dot.edge("activation", "output")

    display.display(dot)

x1_slider = widgets.FloatSlider(min=0, max=1, step=0.05, value=0.85, description='x1 (grade)')
w1_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.8, description='w1')

# Final layout
display.display(x1_slider, w1_slider)

display.display(interactive_output(
    visualize_neuron,
    {
        "x1": x1_slider,
        "w1": w1_slider,
    }
))


You might have been able to find a $w_1$ value which worked for one grade. But as soon as you changed the grade the weight stopped working, and the neuron output the wrong value.

This is where bias comes in, try the same test grading neuron example as before, but now with a bias slider:

In [ ]:
# @title
def visualize_neuron(x1=0.5, w1=0.8, b=0.0):
    """
    Visualize a single neuron with interactive sliders.

    Args:
        x1: Input values
        w1: Weight values
        b: Bias value
    """
    # Calculate neuron values
    input1_weighted = x1 * w1
    weighted_sum = input1_weighted
    total_with_bias = weighted_sum + b

    # Simple tanh activation (approximate calculation)
    activation_output = np.tanh(total_with_bias)

    # Create the visualization
    dot = graphviz.Digraph("Neuron", format="png")
    dot.attr(rankdir="LR", size="8,5")
    dot.attr("node", shape="circle")

    # Input nodes
    dot.node("x1", f"{{ x1 | {x1:.2f} }}", shape="record")

    # Weight edges from inputs to multiplication nodes
    dot.edge("x1", "mult1", label=f"w1={w1:.2f}")

    with dot.subgraph(name="cluster_neuron") as sub_dot:
        sub_dot.attr(style='rounded', color='black', label='', shape='circle')

        # Multiplication nodes
        sub_dot.node("mult1", f"{{ x1*w1 | {input1_weighted:.2f} }}", shape="record")

        # Sum
        sub_dot.node("sum", f"{{ inputs sum | {weighted_sum:.2f} }}", shape="record")

        # Bias node
        sub_dot.node("bias", f"{{ b | {b:.2f} }}", shape="record")

        # Total sum node (with bias)
        sub_dot.node("total", f"{{ total sum | {total_with_bias:.2f} }}", shape="record")

    # Activation node
    dot.node("activation", f"tanh", shape="diamond")
    dot.node("output", f"{{ output | {activation_output:.2f} }}", shape="record")

    # Edges for multiplication results to sum
    dot.edge("mult1", "sum")

    # Edge from sum to total with bias
    dot.edge("sum", "total", label="sum")
    dot.edge("bias", "total", label="bias")

    # Edge from total to activation
    dot.edge("total", "activation")

    # Output label
    dot.edge("activation", "output")

    display.display(dot)

x1_slider = widgets.FloatSlider(min=0, max=1, step=0.05, value=0.85, description='x1 (grade)')
w1_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.8, description='w1')
b_slider = widgets.FloatSlider(min=-1, max=1, step=0.05, value=0.8, description='b')


# Final layout
display.display(x1_slider, w1_slider, b_slider)

display.display(interactive_output(
    visualize_neuron,
    {
        "x1": x1_slider,
        "w1": w1_slider,
        "b": b_slider,
    }
))


Now our neuron can accurately predict if a test is a pass or a fail.

Just like the weights, during the training of a neural networking we try to find the bias value which results in the lowest (best) error.

### You Try - Full Neuron
Here is neuron with all values changable. Try tweaking the inputs, weights, and bias to get different values.

- See how many different ways you can make the neuron outputs $0.1$
- Try only changing the inputs
- Try only changing the weights
- Try only changes the weights and the bias

In [ ]:
# @title
def visualize_neuron(x1=0.5, x2=0.3, w1=0.8, w2=-0.4, b=0.2):
    """
    Visualize a single neuron with interactive sliders.

    Args:
        x1, x2: Input values
        w1, w2: Weight values
        b: Bias value
    """
    # Calculate neuron values
    input1_weighted = x1 * w1
    input2_weighted = x2 * w2
    weighted_sum = input1_weighted + input2_weighted
    total_with_bias = weighted_sum + b

    # Simple tanh activation (approximate calculation)
    activation_output = np.tanh(total_with_bias)

    # Create the visualization
    dot = graphviz.Digraph("Neuron", format="png")
    dot.attr(rankdir="LR", size="8,5")
    dot.attr("node", shape="circle")

    # Input nodes
    dot.node("x1", f"{{ x1 | {x1:.2f} }}", shape="record")
    dot.node("x2", f"{{ x2 | {x2:.2f} }}", shape="record")

    # Weight edges from inputs to multiplication nodes
    dot.edge("x1", "mult1", label=f"w1={w1:.2f}")
    dot.edge("x2", "mult2", label=f"w2={w2:.2f}")

    with dot.subgraph(name="cluster_neuron") as sub_dot:
        sub_dot.attr(style='rounded', color='black', label='', shape='circle')

        # Multiplication nodes
        sub_dot.node("mult1", f"{{ x1*w1 | {input1_weighted:.2f} }}", shape="record")
        sub_dot.node("mult2", f"{{ x2*w2 | {input2_weighted:.2f} }}", shape="record")

        # Sum
        sub_dot.node("sum", f"{{ inputs sum | {weighted_sum:.2f} }}", shape="record")

        # Bias node
        sub_dot.node("bias", f"{{ b | {b:.2f} }}", shape="record")

        # Total sum node (with bias)
        sub_dot.node("total", f"{{ total sum | {total_with_bias:.2f} }}", shape="record")

    # Activation node
    dot.node("activation", f"tanh", shape="diamond")
    dot.node("output", f"{{ output | {activation_output:.2f} }}", shape="record")

    # Edges for multiplication results to sum
    dot.edge("mult1", "sum")
    dot.edge("mult2", "sum")

    # Edge from sum to total with bias
    dot.edge("sum", "total", label="sum")
    dot.edge("bias", "total", label="bias")

    # Edge from total to activation
    dot.edge("total", "activation")

    # Output label
    dot.edge("activation", "output")

    display.display(dot)

x1_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.5, description='x1')
w1_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.8, description='w1')
input1_box = widgets.HBox([x1_slider, w1_slider], layout={'border': '1px solid gray'})

x2_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.3, description='x2')
w2_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=-0.4, description='w2')
input2_box = widgets.HBox([x2_slider, w2_slider], layout={'border': '1px solid gray'})

bias_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0.2, description='bias')

# Organize into input group and bias group
inputs_group = widgets.VBox(
    [
        input1_box,
        input2_box
    ],
    layout={
        'margin-left': '200px',
    },
)
bias_group = widgets.VBox(
    [
        bias_slider,
    ],
    layout={
        'border': '2px solid black',
        'padding': '10px',
    },
)
bias_group = widgets.VBox([bias_slider], layout={'border': '2px solid black', 'padding': '10px'})

# Final layout
display.display(widgets.HBox([
    inputs_group,
    bias_group
]))

display.display(interactive_output(
    visualize_neuron,
    {
        "x1": x1_slider,
        "x2": x2_slider,
        "w1": w1_slider,
        "w2": w2_slider,
        "b": bias_slider,
    }
))


# Check In - Neurons
We just learned when to use weight vs bias to tune the behavior of a neuron. Here are the key points you should know:

- Use weight to change how much the neuron pays attention to a certain input
- Use bias when your input data is offset by a specific amount

## Neural Network
A neural network is created by connecting multiple neurons together. The output of some neurons is passed as the input to other neurons.

![3 layer neural network](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/3-layer-neural-network.jpg?raw=true)

Each circle in the diagram above is a neuron.

- The arrows coming out of the right hand side of neurons are outputs
- The arrows going in to the left hand side of neurons are inputs

To help organize so many neurons we classify them into **layers**. You can see these layers by looking at which neurons are connected to which.

- Neurons within a layer don't connect to other neurons in the same layer
- The inputs of neurons in one layer are the outputs of neurons in the previous layer


### Neural Network Structure
How do we decide what structure to use in neural networks?

All neural networks have at least a few things in common:

- There is an input layer of neurons
  - Number of input neurons = number of inputs
  - Example: If the input is 1 number then there is 1 input neuron
- There is an output layer of neurons
  - Number of output neurons = number of outputs
  - Example: If the output of the model is to predict a single number then there will be 1 output neuron
  - Example: If the output of the model is predicting which of 5 categories the input belongs to, then there are 5 output neurons
- There are typically multiple layers of neurons in between the input and output
  - These are called "hidden layers"

### You Try - Counting Layers and Neurons
Look at this diagram of a neural network:

![Diagram of 4 layer neural network](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/4-layer-neural-network.jpg?raw=true)

Answer the following questions about its structure:

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many layers are there in this neural network?"
answer = "4"
hint = "Count the number of groups of neurons"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons are in the input layer?"
answer = "3"
hint = "Count the number of neurons in the input layer"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons are in the output layer?"
answer = "1"
hint = "Look at the output layer, how many neurons are there?"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many hidden layers are in the neural network?"
answer = "2"
hint = "Count the number of groups of neurons between the input and output layers"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

Let's look at another neural network:

![7 layer neural network](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/7-layer-neural-network.png?raw=true)

What can you tell about this neural network?

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many layers are in this neural network?"
answer = "7"
hint = "Count the number of groups of neurons (groups of circles)"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons are in the output layer??"
answer = "3"
hint = "Count the number of neurons in the right-most output layer"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons are in the biggest hidden layer?"
answer = "6"
hint = "There are several hidden layers, for each layer count how many neurons there are, the answer is the biggest number"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

### You Try - Neural Network Structure
In this example we are trying to build a neural network which tries to predict how many goals a soccer player will score given their:

- Age
- Weight
- Height

Answer these questions:

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons will be in the output layer?"
answer = "1"
hint = "The neural network is trying to predict one number, the number of goals a player may score"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons will there be in the input layer?"
answer = "3"
hint = "Review the factor that we think will contribute to the number of goals, how many factors are there?"

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

Here is another example: Imagine we are building a neural network which determines if an image is of a llama or a duck.

We know that there are 256 input neurons required to take an image as an input.

In [ ]:
# @title
# Display question and have input box for answer, then check if answer is correct
question = "How many neurons will be in the output layer?"
answer = "2"
hint = "When the output of a model is not a simple number, but instead an answer to the question which category is the thing in then the number of outputs is equal to the number of catagories."

question_label = widgets.Label(value=question)
answer_box = widgets.IntText(description="Answer")
answer_button = widgets.Button(description="Check answer")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if str(answer_box.value) == answer:
        # make text green
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

answer_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    answer_box,
    answer_button,
    answer_result,
    hint_label,
]))

### Network Structure - Middle Layers
Now we know that the number of neurons in the input layer and output layers is equal to the number of inputs and outputs.

But what about the number of neurons in the middle "hidden layers"? And how many hidden layers should we have?

This is where it gets really tricky:

- Generally as a rule you need at least one hidden layer
- The more hidden layers you have the more complex decisions your neural network can make
- However, there are disadvantages to having more hidden layers
  - Harder to train
  - Bigger computer required to run the model

It is sort of a trial and error approach. There are some rules you can follow to help find the best number of hidden layers and number of neurons. But we won't go into that here.

In addition, many of the newer models (ChatGPT, ect) use automatic methods of finding the best number of neurons and hidden layers.

### Power in Numbers
With just one neuron the predictions a model can make are not that advanced. However, when we combine hundreds, thousands, or even billions of neurons together the neural networks can start producing very impressive output.

The same is true about neurons in real life:

- A honey bee has around 1 billion neurons
- A dog has around 3 billion neurons
- A human has around 9 billion neurons

As you can see, the simplest creatures have a **ton** of neurons. And as the creatures get smarter they have more neurons.

# Check In - Neural Networks
We just learned about Neural Networks. Here are the key points you should understand:

- Neural Networks are made up of a bunch of neurons
- The outputs of neurons are used as the inputs to other neurons
- We orangize neural networks into layers of neurons
- These layers are organized by their purpose:
  - **Input layer**
    - These neurons are responsible for receiving the input value to the neuron network
    - The inputs to these neurons is the input data
    - You need as many neurons as you have inputs to your network
  - **Hidden layer**
    - This is the real "brain" of the neural network
    - This layer learns the data and patterns
  - **Output layer**
    - These neurons are responsible for outputing the final result
    - The outputs of these neurons are used as the output of the model
    - You need as many neurons as you have outputs in your network

# Tensors
Before we program neural networks we have to learn about "Tensors". It's kind of a weird word, what are tensors?

Well, you actually already know. If I should you this Python code, can you tell me what it is?

In [ ]:
[1, 2, 3, 4, 5, 6, 7]

It's a list! And, it's also a "tensor".

So why do we have two names for a list? And why does one of them, "tensor", sound so intense and crazy?

- Well, machine learning is really just a bunch of math
- A lot of that math was made by... mathematicians
- In the math of neural networks there is a lot of crazy stuff going on and behind the scenes tensors have some cool math you can do with them
- Also I think mathematicians also just wanted to use a cool word like "tensors" to make themselves sound impressive
  - Saying "list" doesn't sound that cool, it sounds like you're going grocery shopping or something

But we don't need to worry about all that. Just remember when you see the word "tensor" it's actually just a list.

Now one cool thing you can do with tensors (and lists) is have lists inside of lists. Check this out:

In [ ]:
list_of_lists = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

for a_list in list_of_lists:
    print(a_list)
    print("The first number in the list inside a list is:", a_list[0])

This list inside a list thing turns out to be really helpful when we're programming neural networks. We'll talk more about that later.

# Check In - Tensors
We just learned that "tensor" is a fancy word for "list". Here are the key points you should know:

- A tensor is a list
- A tensor can have lists of lists

# Programming Neural Networks
Now that we know what a neural network is, let's program one.

Just like our programming before we'll be using a tool named PyTorch. This is a standard tool used within machine learning programming.

## Single Layer
Let's start super simple and just define a single layer of neurons in PyTorch:


In [ ]:
one_layer = nn.Sequential(
    nn.Linear(3, 3),
    nn.Tanh(),
)

Let's break down what's happening here:

- PyTorch provides a set of functions and classes via the name `nn` (which stands for Neural Network)
- To define a layer you make a `nn.Sequential` class
  - The arguments to this class tell PyTorch what our neurons look like
- We tell PyTorch that we want 3 neurons by making a `nn.Linear` class
  - We define the shape of a layer by saying how many inputs and outputs the layer has:
    ```python3
    nn.Linear(number_inputs, number_outputs)
    ```
    (This is different than the input layers and output layers that we were talking about earlier)
  - The number of inputs to the layer is equal to the number of neurons in the previous layer
  - The number of outputs from the layer is equal to the number of neurons in the layer we are defining
  - Since we're just making one layer with no layers before or after it we can just use the number of neurons for both arguments
- Then we tell PyTorch we want to use the $tanh$ activation function

We can pass some data through our neurons and see what they output:

In [ ]:
one_layer = nn.Sequential(
    nn.Linear(3, 3),
    nn.Tanh(),
)

input_data = torch.tensor([1.0, 2.0, 3.0])
one_layer(input_data).detach()

Here you can see we gave $1.0$, $2.0$, and $3.0$ as the inputs to our layer. And we will get 3 numbers out in return.

Try running the code in this cell a few times. You'll notice that the output numbers change every time.

What's going on here?

- PyTorch sets the weights to our neurons randomly every time
- This is actually standard practice when training neural networks
- Setting the weights randomly makes sure that the training process finds the best weights
  - Compared to if they were all set at `1` or something

### Your Turn - Single Layer
Use PyTorch to make a single layer:

- The layer should have 10 neurons in it
- Use the $tanh$ activation function

In [ ]:
# Make a single layer
# 10 neurons
# tanh activation function

## Neural Network
Let's make this Neuron Network from earlier in PyTorch:

![3 layer neural network](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/3-layer-neural-network.jpg?raw=true)

To make this Neural Network in PyTorch we:

1. Define a class called `NeuralNetwork` (The name doesn't really matter)
  - The class should inherit from the `nn.Module` class, this is a PyTorch class which provides a lot of the functionality we need for a neural network
2. In the class's `__init__` function define our model structure
  - PyTorch doesn't actually force you to define your model in any standard way
  - So we can just do what makes sense to us
  - Just like we defined layers above, each layer uses `nn.Sequential` along with `nn.Linear` and `nn.Tanh`
3. Then we define a function in our class which passes a value through all of our layers
  - We must call this function`forward()`
  - The function should take in the input to the neural network
  - It should return the output of the neural network


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # Define each of our layers
        self.input_layer = nn.Sequential(
            nn.Linear(3, 3),
            nn.Tanh(),
        )
        self.hidden_layer = nn.Sequential(
            nn.Linear(3, 4),
            nn.Tanh(),
        )
        self.output_layer = nn.Sequential(
            nn.Linear(4, 2),
            nn.Tanh(),
        )

    def forward(self, input_value):
        # Pass the input_value through all our layers
        input_layer_result = self.input_layer(input_value)
        hidden_layer_result = self.hidden_layer(input_layer_result)
        output_layer_result = self.output_layer(hidden_layer_result)

        return output_layer_result

Here you can see that we store each layer in a class variable (`input_layer`, `hidden_layer`, and `output_layer`).

In the `forward` method we then pass the output of one layer into the next layer.

This approach is very clear, and makes it easy to tell the structure of our neural network. But PyTorch actually makes this even easier to do. The `nn.Sequential` class can define multiple layers at once. Look at this simplified class

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # Define all of our layers at once
        self.layers = nn.Sequential(
            # Input layer
            nn.Linear(3, 3),
            nn.Tanh(),

            # Hidden layer
            nn.Linear(3, 4),
            nn.Tanh(),

            # Output layer
            nn.Linear(4, 2),
            nn.Tanh(),
        )

    def forward(self, input_value):
        # Pass the input_value through all our layers
        return self.layers(input_value)

This `NeuralNetwork` class is exactly the same as our previous one. The main advantage is we can get the result of putting a value through our neural network in 1 line (`self.layers(input_value)`) instead of several lines like before.

## You Try - Neural Network
Try making a PyTorch neural network class for the neural network shown in this photo:

![Diagram of 4 layer neural network](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/4-layer-neural-network.jpg?raw=true)


In [ ]:
# Make a NeuralNetwork class which inherits from the nn.Module
# In the __init__ function don't forget to call super().__init__()
# Define a layers variable using nn.Sequential, nn.Linear, and nn.Tanh
# Define a forward function which takes an input value and passes it through the neural network

# Check In - Defining Neural Networks
We just learned how we can define the structure of a neural network using PyTorch. Let's go over the important facts you should know:

- You make a neural network by creating a class which inherits from PyTorch's `nn.Module`
- You define the structure of your neural network in class variables
- By using `nn.Sequential`, `nn.Linear`, and `nn.Tanh` classes
- You define a function named `forward` to take an input value and run it through your neural network

## Training Neural Networks
Now that we know how to create a neural networking in PyTorch. Let's look at training them. The code to do this actually looks really similar to our earlier code with $y = m \cdot x + b$.

This process is made a lot easier by PyTorch. We don't have to worry about updating the weights or bias ourselves. A lot of this is made easier by the PyTorch `nn.Module` parent class:

- Any class variable you make which defines one or more layers is tracked by `nn.Module`'s functionality
- Since `nn.Module` automatically knows about all your layers it also knows about all your weights and biases
- So PyTorch can automatically update your layers to train them

Take a look at how the `named_parameters()` function which `nn.Module` provides knows about all our layers:

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # Define all of our layers at once
        self.layers = nn.Sequential(
            # Input layer
            nn.Linear(3, 3),
            nn.Tanh(),

            # Hidden layer
            nn.Linear(3, 4),
            nn.Tanh(),

            # Output layer
            nn.Linear(4, 2),
            nn.Tanh(),
        )

    def forward(self, input_value):
        # Pass the input_value through all our layers
        return self.layers(input_value)

model = NeuralNetwork()

print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name}")
    print(f"    {param[:2].detach()} \n")

So PyTorch knows about our neural network's structure, and all its weights and biases. Now we're ready to train.

## Datasets
Now we need something to train our model on:

- Let's make a neural network which can detect if the weather is good or bad
- We will have the inputs:
  - Temperature (Fahrenheit)
  - Humidity (Percentage)
  - Cloud Cover (Percentage, 0% completely sunny, 100% completely cloudy)
- The output will be the probability (Percentage) of the day being a good day (0% is a completely bad day, 100% is a completely good day)

Just like training our $y = m \cdot x + b$ model we need some example data to train our model on:

- We need example input values and their known output values

## Training Data vs Test Data
It's common in neural network training to split our dataset into 3 pieces:

- **Training data:** This is the data we use to calculate the error and update weights and baises
- **Test data:** After every round of training we run the model against this data to see how training is going
  - We don't update the weights or biases based on this
  - But we might make decisions about the training process
    - For example we might decide the model needs more rounds of training
    - Or that we aren't updating the weights by a big enough number every time
    - Or that our neural network structure needs tuning
- **Evaluation data:** Run the model against this data at the **very end** to determine how the final trained model works

Typically we take our one big dataset and split it into these 3 catagories based on these percentages:

- ~85% training data
- ~12.5% test data
- ~12.5% evaluation data

Why do we separate our data into these 3 categories?

- If we didn't separate the data at all then the model would be trained on the same data it's being tested on
- That's equivalent to seeing the answers to the test before taking it
- It wouldn't be fair to say the model can predict anything accurately because it's seen all the data before and had its weights updated based on this

So it makes sense why we wouldn't show the model the answers to the test before giving it the test. But why have a separate test and evaluation dataset?

- Although we may not update weights and biases based on the accuracy of the test dataset
- We still make decisions about the training and structure of our model
- We want to make sure none of those decisions were influenced by the test data
- This is similar to how scientists do double blind experiments
  - Imagine scientists are doing an experiment to see if a new drug helps a rare condition
  - They give some patients the old drug which isn't very good
  - They give some patients the new drug which they think can cure them
  - If the doctor giving the patients the drug knows which drug they're giving they may behave differently
    - The doctor giving the new drug might be really happy and in good spirits, influencing the patient to appear to feel better
    - The doctor giving the old not effective drug might be sad, which might make the patient feel bad and not appear to be doing well
- By saving a dataset just for our final test we make sure we aren't accidentally influenced by anything


### Value Ranges in Neural Networks
Neural networks prefer values between 0.0 and 1.0 when possible.

If your values have a known range then it is best to translate those values between 0.0 and 1.0 before giving them to your neural network. For values that don't have a known smallest or largest value it's fine to keep them as is.

The process of taking a range of values and translating them to be between 0.0 and 1.0 is called **normalization**. To normalize any value just divide it by the largest value in the range.

Here are some examples of times when we do know the range and we should normalize:

- ACT scores (as of 2026)
  - The maximum score is a 36 on your ACTs
  - The minimum score is a 0
  - If our neural network took ACT scores we would want to normalize them
  - To normalize ACT scores we just divide ACT scores by 36 (the maximum score)
  - Ex: ACT score of 22 => $\frac{22}{36} \approx 0.61$
  - Ex: ACT score of 30 => $\frac{30}{36} \approx 0.83$
- Percentages
  - The minimum value of a percentage is 0
  - The maximum value of a percentage is 100
  - We should normalize percentages to be between 0.0 and 1.0
  - To normalize percentages we just divide a percentage by 100
  - Ex: Percentage of 55% => $\frac{55}{100} = 0.55$
  - Ex: Percentage of 99% => $\frac{99}{100} = 0.99$

You need to be careful that values have a known maximum or minimum before normalizing.

- Don't just go and pick the smallest and largest value in your dataset and use those to normalize
- What if after training a user inputs a higher value
  - For example with temperature
  - You could look at weather reports and find the coldest day on earth and the hottest day on earth and then use those values
  - But what if, due to say global warming, the next year there is a new hottest day on earth
  - Then when you go to normalize a value it would be higher than 1.0

### Outputs
The same rule can be applied to our neural network's outputs.

- If you know the minimum and maximum value then normalize it in your dataset
- For example in our good weather / bad weather dataset
  - We should normalize the percentage
  - 0% => bad day => normalized to 0.0
  - 100% => good day => normalized to 1.0

# Check In - Dataset Concepts
We just learned about the best way to structure our datasets. Here are the key things you should know:

- Datasets are a collection of input values and what the output should be
- We should normalize values with known ranges (like test scores or percentages) to numbers between 0.0 and 1.0
- We split our dataset into training, test, and evaluation datasets
  - Training datasets are used to calculate error and update weights and biases during training
  - Test datasets are used to see how our model is improving after each training round
  - Evaluation datasets are used at the very end of our training to see our model's final performance
- We don't want to train on the data we then use to evaluate how well our neural network does

## Creating a Dataset in PyTorch
To create a dataset in Pytorch we:

- Make a class which inherits from the PyTorch `Dataset` class
- We must override 2 specially named functions:
  - `__len__` should return the number of examples in our dataset
  - `__getitem__` should return a single example and its outcome
    - This function takes an `index` argument, just like a list it indicates which item in the dataset to retrieve

Once we do this PyTorch can use our dataset to train our neural network:

In [ ]:
class GoodOrBadWeatherDataset(Dataset):
    def __init__(self):
        # Define a class variable with all our data in it
        # - Temperature (Fahrenheit)
        # - Humidity (Percentage)
        # - Cloud Cover (Percentage, 0% completely sunny, 100% completely cloudy)
        #
        # Also add good_weather which is 1 if the weather is good, 0 if not
        self.data = [
            # Good weather days (sunny, comfortable)
            {"temperature": 72.5, "humidity": 0.45, "cloud_coverage": 0.20, "good_weather": 1.0},
            {"temperature": 75.0, "humidity": 0.50, "cloud_coverage": 0.30, "good_weather": 1.0},
            {"temperature": 78.2, "humidity": 0.42, "cloud_coverage": 0.15, "good_weather": 1.0},
            {"temperature": 71.0, "humidity": 0.48, "cloud_coverage": 0.25, "good_weather": 1.0},
            {"temperature": 76.5, "humidity": 0.55, "cloud_coverage": 0.35, "good_weather": 1.0},
            {"temperature": 73.8, "humidity": 0.40, "cloud_coverage": 0.10, "good_weather": 1.0},
            {"temperature": 79.0, "humidity": 0.44, "cloud_coverage": 0.20, "good_weather": 1.0},
            {"temperature": 74.2, "humidity": 0.52, "cloud_coverage": 0.40, "good_weather": 1.0},
            # Some questionable days (partially cloudy)
            {"temperature": 68.0, "humidity": 0.72, "cloud_coverage": 0.65, "good_weather": 0.0},
            {"temperature": 65.5, "humidity": 0.78, "cloud_coverage": 0.72, "good_weather": 0.0},
            {"temperature": 62.0, "humidity": 0.68, "cloud_coverage": 0.60, "good_weather": 0.0},
            # Bad weather days (rainy, overcast, too hot/cold)
            {"temperature": 58.0, "humidity": 0.85, "cloud_coverage": 0.95, "good_weather": 0.0},
            {"temperature": 55.5, "humidity": 0.88, "cloud_coverage": 1.0, "good_weather": 0.0},
            {"temperature": 92.0, "humidity": 0.65, "cloud_coverage": 0.40, "good_weather": 0.0},
            # More good weather days
            {"temperature": 77.0, "humidity": 0.46, "cloud_coverage": 0.18, "good_weather": 1.0},
            {"temperature": 80.5, "humidity": 0.38, "cloud_coverage": 0.12, "good_weather": 1.0},
            {"temperature": 72.8, "humidity": 0.50, "cloud_coverage": 0.22, "good_weather": 1.0},
            # More bad weather
            {"temperature": 60.2, "humidity": 0.82, "cloud_coverage": 0.88, "good_weather": 0.0},
            {"temperature": 59.0, "humidity": 0.86, "cloud_coverage": 0.92, "good_weather": 0.0},
            # Mixed - some good, some not
            {"temperature": 70.5, "humidity": 0.65, "cloud_coverage": 0.55, "good_weather": 1.0},
            {"temperature": 69.0, "humidity": 0.70, "cloud_coverage": 0.68, "good_weather": 0.0},
            {"temperature": 73.2, "humidity": 0.58, "cloud_coverage": 0.38, "good_weather": 1.0},
            {"temperature": 67.8, "humidity": 0.75, "cloud_coverage": 0.70, "good_weather": 0.0},
            # Final good weather days
            {"temperature": 76.0, "humidity": 0.42, "cloud_coverage": 0.20, "good_weather": 1.0},
            {"temperature": 78.5, "humidity": 0.45, "cloud_coverage": 0.25, "good_weather": 1.0},
            {"temperature": 71.5, "humidity": 0.48, "cloud_coverage": 0.30, "good_weather": 1.0},
            {"temperature": 74.0, "humidity": 0.52, "cloud_coverage": 0.35, "good_weather": 1.0},
            {"temperature": 66.0, "humidity": 0.72, "cloud_coverage": 0.65, "good_weather": 0.0},
            {"temperature": 75.5, "humidity": 0.55, "cloud_coverage": 0.30, "good_weather": 1.0},
            {"temperature": 64.0, "humidity": 0.78, "cloud_coverage": 0.75, "good_weather": 0.0},
            {"temperature": 72.0, "humidity": 0.50, "cloud_coverage": 0.28, "good_weather": 1.0},
        ]


    def __len__(self):
        # Return the number of data points in the dataset
        return len(self.data)

    def __getitem__(self, index):
        # Return the item as a list of values, and then the good_weather value as the desired outcome
        item = self.data[index]
        return (
            torch.tensor([
                item["temperature"],
                item["humidity"],
                item["cloud_coverage"],
            ]),
            torch.tensor([item["good_weather"]])
        )

Now we have a dataset of weather and if the weather on that day is considered "good".

Let's see this on a graph:

In [ ]:
# @title
dataset = GoodOrBadWeatherDataset()

xs = []
ys = []
zs = []
colors = []

for item in iter(dataset):
    xs.append(item[0][0])
    ys.append(item[0][1])
    zs.append(item[0][2])
    colors.append("green" if item[1] == 1 else "red")

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode='markers',
    marker=dict(
        color=colors,
        size=8,
        opacity=0.8
    )
)])

# Update layout with axis labels
fig.update_layout(
    scene=dict(
        xaxis_title='Temperature',
        yaxis_title='Humidity',
        zaxis_title='Cloud Coverage'
    ),
    width=800,
    height=600,
    margin=dict(l=0, r=0, b=0, t=0)
)

# Show the interactive plot
fig.show()


(You can click and drag to re-orient to graph)

Here you can see our dataset graphed in 3d. Each of the 3 axis is one of our metrics:

- Cloud Coverage
  - A percentage from 0.0 to 1.0 (0.0 is 0%, 1.0 is 100%)
- Humidity
- Temperature

The color of the point indicates if the weather is considered good (green) or bad (red).

## Splitting Our Dataset
Now that we have our PyTorch dataset let's split our dataset into training, test, and evaluation groups.

PyTorch provides the very useful [`random_split()`](https://docs.pytorch.org/docs/2.12/data.html#torch.utils.data.random_split) function which we can use to take our data and randomly split it into sub-groups.

This function takes a list of the lengths of the different sub-groups.

- So we need to find the total number of items in our dataset
- Then get the count in each sub-group using percentages

In [ ]:
dataset = GoodOrBadWeatherDataset()

total_items = len(dataset)

training_size = int(total_items * 0.85)
test_size = int(total_items * 0.125)

# Get the remaining number of items
# Since multplying times half numbers could leave some items out if we have an odd or even number of samples
evaluation_size = total_items - (training_size + test_size)

training_dataset, test_dataset, evaluation_dataset = random_split(dataset, [training_size, test_size, evaluation_size])

print(f"total items: {total_items}")
pprint(f"training: {list(training_dataset)}")
pprint(f"test: {list(test_dataset)}")
pprint(f"evaluation: {list(evaluation_dataset)}")

You can see that there aren't a ton of items in our test and evaluation datasets. That's mainly because we don't have a ton of data anyway. In real neural network training you'd have thousands or tens of thousands of items.

# Check In - PyTorch Datasets
We just learned how to create a PyTorch dataset and split it into training, test, and evaluation sub-datasets. Here are the key points you should know:

- Create a PyTorch dataset by making a class which inherits from PyTorch's `Dataset` class
  - Implement the `__len__` function to tell PyTorch how many items are in your dataset
  - Implement the `__getitem__` function to get a specific item in your dataset by its list index
- You can split your dataset into training, test, and evaluation sub-datasets with PyTorch's `random_split` function

## Weather Prediction Neural Network
For our neural network for predicting weather we wil use the following neural network:

In [ ]:
class WeatherNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # Define all of our layers at once
        self.layers = nn.Sequential(
            # Input layer
            # - Temperature
            # - Humidity
            # - Cloud coverage
            nn.Linear(3, 16),
            nn.ReLU(),

            # Hidden layer 1
            nn.Linear(16, 32),
            nn.ReLU(),

            # Hidden layer 2
            nn.Linear(32, 16),
            nn.ReLU(),

            # Output layer
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, input_value):
        # Pass the input_value through all our layers
        return self.layers(input_value)

This neural network has 4 layers:

- Input layer
  - 3 inputs
    - 1 for each of the input values (temperature, humidity, cloud coverage)
  - 16 neurons
  - ReLU activation function
- Hidden layer 1
  - 32 neurons
  - ReLU activation function
- Hidden layer 2
  - 16 neurons
  - ReLU activation function
- Output layer
  - 1 neuron
  - Sigmoid activation

We have 2 hidden layers and the input layer.

- We use the ReLU activation function for these neurons since we want big values like the temperature to get preserved between the layers
  - Since Tanh and Sigmoid take large values and squash them down to $1$
  - We want an activation function like ReLU which allows these large values to exist

For the output layer we use the Sigmoid activation function:

- This is because we want an output between 0.0 and 1.0
  - Representing a probability of the day's weather being good
  - 0.0 => 0% => no chance the day's weather will be good
  - 1.0 => 100% => guarantee the day's weather will be good
  - The Sigmoid activation function takes any value and squishes it into this 0.0 to 1.0 range

## Training with Our Dataset
The process of training a neural network with our dataset is pretty easy with PyTorch. You might be surprised to learn that it only requires a few lines of Python.

- For each item in our training dataset
  - Give the item to our neural network and see if it predicts good or bad weather
  - Calculate the error based on the prediction
  - Update the weights and biases of our model (based on the derivative)
    - Since PyTorch knows about all the weights and biases in all our layers it can help us out and update all our weights for us
    - PyTorch calls the thing which helps with this step an **optimizer**
    - We will use the "Adam" optimizer
- See how accurate our model was by running it by the test data
  - Don't calculate error or update weights based on this
  - Just helpful to see how our model is improving (or not 😬)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Setup the neural network
model = WeatherNeuralNetwork().to(device)

# Define how we're going to train the model
learning_rate = 0.0005
num_training_rounds = 500

error_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

stop_training_below_error = 0.05

def training_loop(model, dataset, optimizer, error_fn):
    # Function which runs our model through a dataset and returns the error
    total_error = 0.0

    for input_data, expected_output in iter(dataset):
        # See what the model predicts
        prediction = model(input_data.to(device))

        # Calculate the error (how close the prediction was to the correct result)
        error = error_fn(prediction, expected_output.to(device))

        total_error = total_error + error

        # If an optimizer is specified then updates the model parameters using it
        if optimizer is not None:
            # Get the derivative of all the weights and biases in relation to the error equation
            error.backward()

            # Update the weights and biases by a small amount in the correct direction
            optimizer.step()

            # Reset the derivatives to zero so we can re-calculate them for the next dataset item
            optimizer.zero_grad()

    return total_error


# Training loop
for training_round in range(num_training_rounds):
    # Update the parameters of our model with the training dataset
    # Tell our model it's about to be trained
    model.train()

    training_loop(
        model=model,
        dataset=training_dataset,
        optimizer=optimizer,
        error_fn=error_fn,
      )

    # Use the test dataset to determine how training is going
    # Tell our model it's about to be evaluated
    model.eval()

    test_error = training_loop(
        model=model,
        dataset=test_dataset,
        optimizer=None, # Do not provide an optimizer, so parameters are not updated
        error_fn=error_fn,
      )

    # Stop training if our error is low enough
    if test_error <= stop_training_below_error:
        print(f"In training round {training_round} we achieved test error {test_error}, exiting early")
        break

    # Print our progress every 100 training rounds
    if training_round % 100 == 0 or training_round == num_training_rounds - 1:
        print(f"Training round {training_round}, test error {test_error}")

# Finally see how accurate our model is via the evaluation dataset
# Tell our model it's about to be evaluated
model.eval()

eval_error = training_loop(
    model=model,
    dataset=evaluation_dataset,
    optimizer=None, # Do not provide an optimizer, so parameters are not updated
    error_fn=error_fn,
)

print(f"After {training_round+1} training rounds the model has an evaluation error of {eval_error}")

Great, now our model has been trained. You can see the final test error and the evaluation error are slightly different. However, they should still be fairly similar.

- This is because of our small evaluation dataset (of only 2 items)
- It's possible those 2 items are harder or easier for our model to predict than the last run of the test data
- The evaluation error and the test error won't ever be exactly the same
- But as dataset size increases they should be much more similar

Now let's use our model! It's as easy as just calling our model like `model(input)`:

In [ ]:
# Input parameters
temperature = 60
humidity = 0.6
cloud_coverage = 0.2

# Call the trained model
output = model(torch.tensor([
    temperature,
    humidity,
    cloud_coverage
]).to(device))

# Interpret the results
# The output is a probability (between 0.0 and 1.0 => 0% and 100%) of the day's weather being good
print(f"Probability of it being a good day: {output.item()}")
if output >= 0.5:
    print("It will be a good weather day")
else:
    print("It will be a bad weather day")

Try playing around with different `temperature` (in Farenheight), `humidity`(0.0 to 1.0), and `cloud_coverage` (0.0 is no clouds, 1.0 is all clouds) input values.

See how well the model predicts the output.

# Check In - Weather Neural Network Training
We just created and trained a neural network to predict if the day's weather will be good or bad. Here are the key points you should know:

- We picked our layer activation functions based on which values we wanted to see in the layer's output
  - ReLU in hidden to preserve big temperature values
  - Sigmoid in output so we only get values between 0.0 and 1.0
- To train our neural network in PyTorch
  - We get the error of our model by comparing its prediction to the expected output
  - We use a PyTorch **optimizer** to update our model's weights and biases based on the error

# Bonus - Model and Hyperparameter Tuning
At this point we have a working neural network. If you want to learn a bit more check out this section. Otherwise move on to the next notebook.

We can change the structure of our neural network and how we train it to try and achieve better results and faster training. There are a few things we can change:

- Model structure
  - How many layers
  - How many neurons in each layer
  - Activation function for each layer
- Training hyperparameters
  - Optimizer
  - Learning rate
  - Training rounds



## Training Hyperparameters
Let's start by talking about "hyperparameters". What are they?

- Hyperparameters indicate how we train the model
- For example: The number of training loops we run our model through (aka the number of times we update our model's weights and biases by showing it each item in the training dataset)
- These are different from the **parameters** of a neural network like the weights and biases in neurons



### Learning Rate
The learning rate is how much weights and biases are changed during each step of training. The learning rate is a hyperparameter.

- A high learning rate will result weights and biases changing a lot
- A small learning rate will result in smaller changes to the weights and biases

The learning rate also changes how fast the model is trained:

- A high learning rate results in lower error **faster**
- A small learning rate results in a lower error **slower**

So why not just always go with the biggest learning rate to get the model trained faster?

- A high learning rate has the risk of changing the weights and biases too much and missing the ideal value
  - Imagine a weight is set at $0.6$
  - We magically know (for the sake of this example) that the ideal value for this weight is at $0.5$
  - If we have a high learning rate the weight might be updated to $0.2$
  - And the error will suddenly get worse!
- It's a balance between finding a learning rate that:
    - Isn't so low that training takes forever
    - Isn't so high that we update weights and biases too much and miss their ideal values

### Optimizer
The optimizer is the part of PyTorch which updates neuron weights and biases based on the derivative of the error.

The optimizer is given a **learning rate** value and updates the weights and biases proportionally to this learning rate.

- The optimizer is the thing which updates the weights and biases
  - A lot if the learning rate is high
  - A little if the learning rate is low

There are different strategies for how we update these weights and biases. Which optimizer strategy you choose is a hyperparameter.

- Stochastic Gradient Descent (aka SDG): This updates model parameters proportionally to the derivative of each weight and bias
  - This is what we did when updating the $m$ and $b$ values of our $y = m \cdot x + b$ model with plant growth earlier
  - This is the most simple way we can update weights and biases
  - Does not take into account how much error changed when we updated weights and biases previously by a similar amount
- ADaptive Moment estimation (aka Adam): This updates model parameters using derivatives plus the past history of how previous changes affected the error
  - Remembers how much it changed each weight and bias last training loop
  - It will update each weight or bias a more or less based on how much the error changed
  - So if it changed a weight a small amount and the error decreased only by a bit, next time it will try changing that weight by a lot

PyTorch provides a bunch of other optimizers, [see them here](https://docs.pytorch.org/docs/main/optim.html#algorithms).

### Number of Training Rounds
You can't keep training a neural network forever. At some point you have to call it quits and use the neural network you have.

Deciding when to stop training is a hyperparameter.

- Stop too soon and your neural network's error may be too high
- Keep training too long and you probably won't get your error much lower
- Worst case you train your model for too long and it only memorizes your data
  - This is called overfitting
  - In this case if you show your neural network completely new data it will make horrible predictions (despite having low error)

## You Try - Training Hyperparameters
In the code we used to train our model above try modifying the training hyperparameters to see how that affects the training process:

- Learning rate: Change the `learning_rate` variable
  - Double its value and see what happens
  - Half its value and see what happens
  - Make the value really big and see what happens
  - Look at what happens to the test error every 100 rounds
- Number of training rounds: Change the `num_training_rounds` variable
  - Half the number of training rounds and see what happens
  - Double the number of training rounds and see what happens
  - Notice how the test error changes
- Optimizer: Change the `optimizer` variable to a different class
  - Try the `torch.optim.SGD` optimizer instead of `torch.optim.Adam`
  - Does the final evaluation error get as low in the same number of training rounds
  - Do you have to modify the `learning_rate` to get good results?
  - Do you have to increase or decrease the `num_training_rounds` to get similar results?
  - Are you ever able to get as good results?

Play around with all these values and see what combinations work. Can you get better results than what we started with? Why do you think each of these numbers affects how good our end model is?

## Model Structure
The structure of your model can have a big affect on how well it performs:

- Using the wrong activation functions
  - Can make it so your model never gets a low error
  - Can make it take way longer to train
- Having the wrong number of hidden layers and or neurons
  - Too few and your neural network may not be able to learn the patterns of your data
  - Too many and your neural network may become overfit

## You Try - Model Structure
Try changing the structure of our `WeatherNeuralNetwork`.

- Make sure the first layer is always `nn.Linear(3, ...)`
  - The first value must be `3` in order to receive our input values correctly
- Make sure  the last layer is always `nn.Linear(..., 1)`
  - The second value must be `1` in order to output our weather probability
- Try changing the number of neurons in our layers
  - Remember the second argument of `nn.Linear` in the previous layer should be the first argument of `nn.Linear` in the next layer
  - ```python
    nn.Linear(can_be_different, SAME)
    # ... activation function

    nn.Linear(SAME, can_be_different)
    # ... activation function
    ```
- Try changing the number of hidden layers
- Try changing the activation functions for all the layers

Make sure after you make each of these changes that you run the code in the cell which defines the class, and then also re-run the code in the training cell.

Ask yourself:

- How does the model performance improve or get worse as each of these changes are made?
- Do you have to change any training hyperparameters to make training work?
- Why would each of these changes result in better or worse performance?